# SLOE tutorial: Sonifying whistler-mode chorus waves in Earth's magnetosphere

In this tutorial we'll use the **SLOE** framework — *Spectraliser Libraries for Offline Exploration* — to turn a real **wave power spectrogram** into sound. SLOE is the SoRBET wrapper around STRAUSS's `Spectralizer` generator, which is a really cool bit of kit: it takes a frequency spectrum from your data and maps it directly onto an *audible* frequency range using an inverse FFT. So the *shape* of your scientific spectrum literally becomes the *timbre* of the sound. Peaks in your data become peaks in the sound — your ears pick up on spectral structure that's quite easy to miss in a colour plot.

This is conceptually different from ManGOE (which played notes whose volume / cutoff / pitch were driven by your data). With SLOE, the spectrum **is** the sound. Cool, right?

Today we'll be sonifying **whistler-mode chorus (WMC) waves** measured in the magnetosphere on **1 March 2013**. Chorus waves are some of the most exciting waves in near-Earth space — they sit in the **whistler band**, which spans frequencies from a fraction of the local electron gyrofrequency $f_{ce}$ down to the very low kHz, and they're famous for accelerating electrons up to relativistic energies in the radiation belts. They're called "chorus" because when you sonify them (which people have been doing since the days of analogue radio receivers!) they sound a bit like a dawn chorus of birds — a flurry of rising tones.

In our data, the frequency axis is given in **units of $f/f_{ce}$** (so a value of 0.5 means "half the local electron gyrofrequency"), and the time axis is in **seconds**. We're looking at a 6-second snapshot — which sounds short, but at chorus frequencies that's *plenty* of time for hundreds of individual chorus elements to sweep past.

The Spectralizer is going to take this $f/f_{ce}$-scaled wave power and stretch it into our audible band, so we can hear the chorus structure directly.

---


## Setup and Imports

First, let's import everything we need. This should be familiar from previous sessions.


In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

# Point to relevant subroutines and data folders
import sys
sys.path.insert(0, '../subroutines/')
sys.path.insert(0, '../Data/')

# Import the SLOE sonification routines
from SLOE import SpectrumSonification, SpectrogramSonification, SLOE_Object


## Eyes on the Spectrum — Visualising the Data

Before we make anything audible, let's actually look at what we're sonifying.

The data file is a CSV with a slightly unusual layout — and it's worth understanding it because it's exactly how a lot of real wave spectrogram data gets shipped around:

- The **first column** is `time` (in **seconds** from the start of the interval).
- The **first row** (from the second column onwards) holds the **frequency bin centres**, given in **units of $f/f_{ce}$** — i.e. as a fraction of the local electron gyrofrequency. (So `0.5` means "half $f_{ce}$".) This is a really common normalisation for whistler-band data because chorus has a characteristic structure relative to $f_{ce}$ — it tends to live between roughly $0.1\,f_{ce}$ and $0.5\,f_{ce}$, with a famous "gap" near $0.5\,f_{ce}$.
- Everything else in the table is the **power spectral density** at that time and frequency.

So the file is a 2D spectrogram with axis labels baked into the first row and first column. We'll load it carefully and split it back out into three pieces: a 1D time array, a 1D frequency array (in $f/f_{ce}$), and a 2D spectrogram.


In [ ]:
# Load the file. The first row is a header (time, then frequency bin centres),
# so we read the header separately to get the frequency axis.
filename = '../Data/PowerSpectrum_20130301_t02_0_73.csv'

# Read just the header line to grab the frequency bin centres
with open(filename, 'r') as f:
    header = f.readline().strip().split(',')

# Header is ['time', f0, f1, f2, ...] — drop 'time' and convert the rest to floats.
# These are normalised frequencies, in units of f/f_ce (the local electron gyrofrequency).
f_norm = np.array([float(s) for s in header[1:]])

# Now load the numerical body. First column is time (seconds), the rest are PSD values.
data = np.genfromtxt(filename, delimiter=',', skip_header=1)
time_s = data[:, 0]              # time axis, in seconds from start of interval
spectrogram = data[:, 1:]        # shape: (n_time, n_freq)

print(f"time array : {time_s.shape}, range {time_s.min():.2f}–{time_s.max():.2f} s")
print(f"freq array : {f_norm.shape}, range {f_norm.min():.3f}–{f_norm.max():.3f} f/f_ce")
print(f"spectrogram: {spectrogram.shape} (time x freq)")


Now let's plot it. Wave power spectra cover several orders of magnitude, so we'll plot the **logarithm** of the power — this is by far the most common way of looking at this kind of data, and the one your eye is best at reading.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Use a small floor before taking log10 so the zeros (data gaps) don't blow up
floor = 1e-2
log_psd = np.log10(np.maximum(spectrogram, floor))

# pcolormesh wants edges, but passing centres works fine for a quick look.
mesh = ax.pcolormesh(time_s, f_norm, log_psd.T, shading='auto', cmap='viridis')

# Reference lines at the chorus band edges (typical: 0.1 f_ce and 0.5 f_ce, with a gap at 0.5)
ax.axhline(0.1, color='white', linestyle=':', linewidth=0.8, alpha=0.7)
ax.axhline(0.5, color='white', linestyle=':', linewidth=0.8, alpha=0.7)

ax.set_xlabel('Time [s]')
ax.set_ylabel(r'Frequency [$f/f_{ce}$]')
ax.set_title('Whistler-mode chorus power spectrogram (1 March 2013)')
cb = plt.colorbar(mesh, ax=ax)
cb.set_label(r'log$_{10}$(PSD)')
plt.tight_layout()
plt.show()


### Something to Think About
*Take a gander at the spectrogram. Where is most of the wave power concentrated — is it sitting in the lower-band ($0.1 < f/f_{ce} < 0.5$) where chorus typically lives, or is there activity elsewhere? Can you spot the famous "power gap" at $f/f_{ce} \approx 0.5$ — the dotted line in the middle of your plot? Are there bright streaks that drift in frequency over a fraction of a second — those would be individual **chorus elements**.*

*A handy thing to remember: silence in a sonification can mean genuine quiet in the data, **or** it can mean missing data. Both are scientifically interesting. As you'll hear shortly, your ears are very good at noticing both.*

---


## Our First Sonification — One Slice in Time

Let's start with the simplest possible thing: pick **a single moment in time**, take the spectrum at that instant (just one row of our 2D array), and turn it into a sustained tone. That's exactly what `SpectrumSonification` does — you give it a 1D array of power values ordered low-frequency-to-high, and it renders a tone whose timbre **is** that spectrum.

Try changing `time_index` and listening to how the sound changes between a quiet interval and a loud one.


In [ ]:
# Pick one row of the spectrogram = one spectrum at one time
time_index = 200                      # try changing me!
spectrum_slice = spectrogram[time_index, :]

print(f"This is the spectrum at t = {time_s[time_index]:.3f} s")
print(f"Spectrum shape: {spectrum_slice.shape}  (1D — good!)")

# Quick plot of the slice we're about to sonify
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(f_norm, spectrum_slice, 'k-')
ax.axvline(0.5, color='grey', linestyle=':', linewidth=0.8)
ax.set_xlabel(r'Frequency [$f/f_{ce}$]')
ax.set_ylabel('Power spectral density')
ax.set_title(f'Spectrum we are about to sonify (t = {time_s[time_index]:.3f} s)')
plt.tight_layout()
plt.show()


In [ ]:
# Set our notes, length and sound system
notes = [["A3"]]            # a single sustained note — the spectrum gives it its colour
length = 5                  # 5 seconds is plenty for a static tone
system = 'mono'

# Render it!
soni = SpectrumSonification(notes, length, spectrum_slice,
                            min_freq=80, max_freq=2000)

print("Sonification rendered! Have a listen:")
dobj = soni.notebook_display()


### Something to Think About
*What does this tone sound like to you — bright? dull? hollow? buzzy? Now go back, change `time_index` to a moment where the spectrogram looks visually quiet (lots of dark/zero patches), and try one where it looks bright. Does the timbre match what you expected from the colour plot?*

*Also try playing with `min_freq` and `max_freq` — these set the audible band that your ULF spectrum gets stretched onto. A wider range gives a richer, more "open" sound; a narrower range concentrates the spectral features into a smaller patch of your hearing.*

---


## The Full Story — Sonifying the Whole Spectrogram

A single spectrum is a single moment. But the whole point of a spectrogram is that the spectral shape **evolves**. `SpectrogramSonification` takes the full 2D array and cross-fades smoothly between rows, so what you hear is the timbre changing as the wave activity changes — for real, in time, with no other mappings layered on.

Our data covers 6 seconds, and that's already a comfortable listening duration — so unlike the ManGOE tutorial (where we squashed days into seconds), here we can play the spectrogram out at roughly its native cadence. That's quite nice for chorus: you should hear individual rising tones if they're there.


In [ ]:
notes = [["A3"]]
length = 6                    # match the data duration: 6 s of data → 6 s of audio
system = 'mono'

soni = SpectrogramSonification(notes, length, spectrogram,
                               min_freq=80, max_freq=2000)

print("Spectrogram sonified! Press play:")
dobj = soni.notebook_display()


### Something to Think About
*Listen for individual rising tones — these would be **chorus elements**, sweeping up in frequency over a few tenths of a second. Listen for moments when the sound goes quieter or vanishes — do they line up with the dark patches in the spectrogram you plotted earlier?*

*This is the diagnostic side of sonification: silence and patchiness in the audio directly reflects silence and patchiness in the data. That can be a feature (you really notice the data gaps!), or a hint that something needs cleaning up before science gets done.*

*Also — the name "chorus" really earns itself once you listen. People who studied these waves with VLF radio receivers in the 1960s named them after birdsong because that's exactly what they sound like. You're now hearing the same thing, just from a digital spectrogram instead of an analogue receiver.*

---


## Layering on Extra Data — `SLOE_Object`

So far we've let the spectrum do all the work. But often we want to layer **another** scientific quantity on top — say, an overall measure of how active things are, that drives volume or a filter cutoff. `SLOE_Object` is the SLOE-equivalent of `ManGOE_Object` from the previous tutorial: you pass a dictionary of mappings, and SLOE builds the sonification for you.

The one mapping you **must** include is `'spectrum'`. Everything else is optional.

**Available sound properties (besides `'spectrum'`):**
- `'volume'` — higher values → louder
- `'cutoff'` — controls a low-pass filter (higher values → brighter / sharper sound)
- `'pitch_shift'` — shifts the pitch up or down by half an octave
- `'azimuth'` — pans the sound left to right (0 = full left, 1 = full right)

To get a 1D quantity from our 2D spectrogram to drive these, we'll compute the **total wave power** at each time step (integrating across frequencies). High total power = lots of wave activity at that moment.


In [ ]:
# Total wave power as a function of time = sum of PSD across the frequency axis
total_power = spectrogram.sum(axis=1)
print(f"total_power shape: {total_power.shape}  (1D, length matches time axis — good!)")

# Quick plot so we can see what we're about to map
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time_s, total_power, 'k-')
ax.set_xlabel('Time [s]')
ax.set_ylabel('Integrated wave power')
ax.set_title('Total wave power across all frequencies')
plt.tight_layout()
plt.show()


Now let's set up the mappings. As with ManGOE, you build a dictionary where the **keys are sound properties** (in quotes) and the **values are data arrays** (no quotes). Set anything to `None` if you don't want to use it.


In [ ]:
# Set up your mappings here

mappings = {
    'spectrum'   : spectrogram,        # REQUIRED — the spectral data we are sonifying
    'volume'     : total_power,        # Louder when there's more wave activity
    'cutoff'     : None,               # Try setting this to total_power as well!
    'pitch_shift': None,               # Try -np.sign(...) of something for sudden octave kicks
    'azimuth'    : None,               # Try time_s to pan left → right as time progresses
}

# Notes, duration, sound system
notes  = [["A3"]]
length = 6
system = 'mono'

# Summary
print("Sonification data set!")
for prop, data in mappings.items():
    status = '✅' if data is not None else 'None'
    print(f"  {prop:.<20s} {status}")


In [ ]:
soni = SLOE_Object(notes, length, mappings,
                   min_freq=80, max_freq=2000,
                   system=system)

print("Sonification rendered! Have a listen:")
dobj = soni.notebook_display()


### Something to Think About
*Compare this to the plain spectrogram sonification we did just before. Does the `volume` mapping make the busy moments more obvious? Does it make the quiet moments **too** quiet — is there structure you wanted to hear that's now buried below your speakers' threshold?*

*This is one of the trade-offs of sonification: every time you map a new variable to a sound property, you're emphasising one thing and de-emphasising another. There's no single right answer — the best mapping depends on what scientific question you're trying to answer with your ears.*

---


## Saving Your Sonification

Once you've found a mapping you're happy with, save the audio file:


In [ ]:
output_filename = 'chorus_spectrogram_sonification.wav'
soni.save(output_filename)
print(f"Saved to {output_filename}")


## Wrapping Up

Hopefully this tutorial has given you a feel for what the **Spectralizer** route through STRAUSS does, and how SLOE makes it quick to use:

- `SpectrumSonification` — single 1D spectrum → a static tone whose timbre *is* the spectrum
- `SpectrogramSonification` — 2D spectrogram → a tone whose timbre evolves with time
- `SLOE_Object` — same as above, plus extra data streams driving volume, cutoff, pitch_shift, panning, etc.

The big thing to take away is that the Spectralizer doesn't just *represent* spectral structure — it actually *plays it back* as audio. Peaks and bands in your data become peaks and bands in the timbre. That's a really direct way of letting your ears pattern-match on something they'd never normally have access to (chorus waves, in this case, which sit at frequencies far above what we can hear directly *and* at amplitudes only a spacecraft in the magnetosphere can measure).

If you want to keep going, try:
- swapping in a different time interval / a different chorus event
- mapping `azimuth` to `time_s` so the chorus burst sweeps from left ear to right ear
- mapping the same quantity to **two** sound properties at once and seeing how it feels
- making it bad on purpose and figuring out *why* it's bad — that's how intuition gets built
